# Phase 7 — Quatre relevés à la fois

## Objectifs

- Relancer le montage convolutif de la phase 6 avec un lot de 4 relevés, sans rien changer d'autre,
  et observer ce qui se passe.
- Identifier précisément ce qui, dans ce montage, dépend des *autres* relevés du même lot — une
  dépendance qui n'aurait jamais dû exister.
- Corriger en modifiant le **modèle**, pas le lot, puis vérifier que la correction ne coûte rien
  quand le lot redevient grand.


## Note de budget de calcul

Un lot de 4 relevés sur les 58 541 exemples d'entraînement complets représente environ 14 600
itérations par époque. Un chronométrage préalable (mêmes couches, mêmes dimensions que la phase 6,
lot de 4) donne ~77 ms par itération sur cette machine, soit **environ 19 minutes par époque** rien
que pour l'entraînement — plusieurs heures pour une comparaison avant/après avec arrêt anticipé. Ce
n'est pas raisonnable sur une machine sans accélérateur.

**Décision, écrite avant toute mesure** : les expériences de cette phase tournent sur un
sous-échantillon stratifié fixe de 3 000 relevés d'entraînement et 900 relevés de validation, tiré une
seule fois depuis la même découpe que la phase 6 (même graine, mêmes classes). Le sous-échantillonnage
est strictement identique pour l'essai « avant correction », l'essai « après correction » et le
contrôle « après correction, lot de 512 » : c'est la taille de lot qui varie, jamais les données. Le
score de la phase 6 (54,98 % sur le jeu complet) reste la référence citée, mais la comparaison directe
et quantitative de cette phase se fait entre ses propres essais, entraînés sur le même sous-ensemble.


## 1. Imports

In [ ]:
from pathlib import Path
import csv
import random
import re
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset


## 2. Configuration et reproductibilité (identique à la phase 6)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

URL_DATA = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-complete-geocoded-time-standardized.csv"
)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE7_DIR = OUTPUT_DIR / "phase_7_quatre_releves_a_la_fois"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PHASE7_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
CONV_CHANNELS = 96
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4, 8, 16]
HIDDEN_DIM = 128
DROPOUT = 0.30
LEARNING_RATE = 0.003
WEIGHT_DECAY = 0.0001
N_EPOCHS_MAX = 10
PATIENCE = 3

BATCH_SIZE_PANNE = 4
BATCH_SIZE_PHASE6 = 512
TAILLE_SOUS_ECHANTILLON_TRAIN = 3000
TAILLE_SOUS_ECHANTILLON_VAL = 900


## 3. Téléchargement, préparation et découpe (identiques à la phase 6)

In [ ]:
if not DATA_PATH.exists():
    print("Téléchargement du fichier...")
    urllib.request.urlretrieve(URL_DATA, DATA_PATH)
else:
    print(f"Fichier déjà disponible : {DATA_PATH}")

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")

df_avant_filtre_classes_rares = df.loc[
    ~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide
].copy()

compte_classes = df_avant_filtre_classes_rares["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index

df_modele = df_avant_filtre_classes_rares.loc[
    df_avant_filtre_classes_rares["shape_model"].isin(classes_conservees)
].copy()

X = df_modele["comments_clean"].copy()
y = df_modele["shape_model"].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y,
)

def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

vocabulaire = {"<PAD>": 0, "<UNK>": 1}
for texte in X_train:
    for token in tokenizer(texte):
        if token not in vocabulaire:
            vocabulaire[token] = len(vocabulaire)

label_encoder = LabelEncoder()
y_train_ids_complet = label_encoder.fit_transform(y_train)
y_val_ids_complet = label_encoder.transform(y_val)
NOMBRE_CLASSES = len(label_encoder.classes_)

longueurs_tokens = df_modele["comments_clean"].apply(lambda t: len(tokenizer(t)))
MAX_LEN = int(longueurs_tokens.max())

print(f"Vocabulaire : {len(vocabulaire)} | classes : {NOMBRE_CLASSES} | MAX_LEN : {MAX_LEN}")


## 4. Sous-échantillon fixe pour cette phase (stratifié, même graine)

In [ ]:
X_train_reset = X_train.reset_index(drop=True)
X_val_reset = X_val.reset_index(drop=True)

_, X_train_sous, _, y_train_sous = train_test_split(
    X_train_reset, y_train_ids_complet,
    test_size=TAILLE_SOUS_ECHANTILLON_TRAIN, random_state=SEED,
    stratify=y_train_ids_complet,
)
_, X_val_sous, _, y_val_sous = train_test_split(
    X_val_reset, y_val_ids_complet,
    test_size=TAILLE_SOUS_ECHANTILLON_VAL, random_state=SEED,
    stratify=y_val_ids_complet,
)

print(f"Sous-échantillon train : {len(X_train_sous)} | validation : {len(X_val_sous)}")


## 5. Jeu de données et modèle (le modèle de la phase 6, `norme` interchangeable)

In [ ]:
class DatasetConvolutif(Dataset):
    def __init__(self, textes, labels, vocabulaire, max_len):
        self.labels = list(labels)
        self.ids = []
        self.masques = []
        for texte in textes:
            tokens = tokenizer(texte)[:max_len]
            ids = [vocabulaire.get(t, vocabulaire["<UNK>"]) for t in tokens]
            longueur = len(ids)
            if longueur == 0:
                ids = [vocabulaire["<UNK>"]]
                longueur = 1
            masque = [1.0] * longueur + [0.0] * (max_len - longueur)
            ids = ids + [vocabulaire["<PAD>"]] * (max_len - longueur)
            self.ids.append(torch.tensor(ids, dtype=torch.long))
            self.masques.append(torch.tensor(masque, dtype=torch.float32))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.ids[index], self.masques[index], int(self.labels[index])

dataset_train_sous = DatasetConvolutif(X_train_sous, y_train_sous, vocabulaire, MAX_LEN)
dataset_val_sous = DatasetConvolutif(X_val_sous, y_val_sous, vocabulaire, MAX_LEN)

class BlocConvResiduel(nn.Module):
    """Identique a la phase 6, sauf que la couche de normalisation est injectee :
    c'est le seul point qui change entre 'avant' et 'apres' correction."""
    def __init__(self, canaux, kernel_size, dilation, fabrique_norme):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(canaux, canaux, kernel_size, dilation=dilation, padding=padding)
        self.norme = fabrique_norme(canaux)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        sortie = self.conv(x)
        sortie = self.norme(sortie)
        sortie = self.activation(sortie)
        sortie = self.dropout(sortie)
        return x + sortie

class ClassifieurConvolutif(nn.Module):
    def __init__(self, taille_vocabulaire, nombre_classes, canaux, kernel_size, dilations, hidden_dim, fabrique_norme):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocabulaire, canaux, padding_idx=0)
        self.blocs = nn.ModuleList([
            BlocConvResiduel(canaux, kernel_size, dilation, fabrique_norme) for dilation in dilations
        ])
        self.tete = nn.Sequential(
            nn.Linear(canaux, hidden_dim),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, nombre_classes),
        )

    def forward(self, ids, masque):
        x = self.embedding(ids).transpose(1, 2)
        for bloc in self.blocs:
            x = bloc(x)
        masque_etendu = masque.unsqueeze(1)
        somme = (x * masque_etendu).sum(dim=2)
        compte = masque_etendu.sum(dim=2).clamp(min=1)
        pool_moyenne = somme / compte
        return self.tete(pool_moyenne)

def fabrique_batchnorm(canaux):
    return nn.BatchNorm1d(canaux)  # le montage de la phase 6, tel quel

def fabrique_groupnorm(canaux, groupes=8):
    return nn.GroupNorm(groupes, canaux)  # statistiques calculees par exemple, jamais entre exemples du lot


## 6. Outil de mesure commun

In [ ]:
def entrainer_et_mesurer(nom, fabrique_norme, batch_size, n_epochs=N_EPOCHS_MAX, patience=PATIENCE):
    torch.manual_seed(SEED)
    loader_train = DataLoader(dataset_train_sous, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(dataset_val_sous, batch_size=batch_size, shuffle=False)

    modele = ClassifieurConvolutif(
        len(vocabulaire), NOMBRE_CLASSES, CONV_CHANNELS, KERNEL_SIZE, DILATIONS, HIDDEN_DIM, fabrique_norme,
    ).to(DEVICE)
    optimiseur = torch.optim.AdamW(modele.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    fonction_perte = nn.CrossEntropyLoss()

    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    meilleure_perte_val = float("inf")
    meilleur_etat = None
    epochs_sans_amelioration = 0

    debut = time.perf_counter()
    for epoch in range(1, n_epochs + 1):
        modele.train()
        perte_totale, nombre_exemples = 0.0, 0
        for ids, masques, labels in loader_train:
            optimiseur.zero_grad()
            logits = modele(ids, masques)
            perte = fonction_perte(logits, labels)
            perte.backward()
            optimiseur.step()
            perte_totale += perte.item() * len(labels)
            nombre_exemples += len(labels)
        perte_train = perte_totale / nombre_exemples

        modele.eval()
        perte_val_totale, nombre_val, predictions, labels_reels = 0.0, 0, [], []
        with torch.no_grad():
            for ids, masques, labels in loader_val:
                logits = modele(ids, masques)
                perte = fonction_perte(logits, labels)
                perte_val_totale += perte.item() * len(labels)
                nombre_val += len(labels)
                predictions.extend(logits.argmax(dim=1).tolist())
                labels_reels.extend(labels.tolist())
        perte_val = perte_val_totale / nombre_val
        acc_val = accuracy_score(labels_reels, predictions)

        historique["epoch"].append(epoch)
        historique["train_loss"].append(perte_train)
        historique["val_loss"].append(perte_val)
        historique["val_acc"].append(acc_val)

        print(f"[{nom}] epoch {epoch:02d} | train={perte_train:.4f} | val={perte_val:.4f} | acc={acc_val:.2%}")

        if perte_val < meilleure_perte_val:
            meilleure_perte_val = perte_val
            meilleur_etat = {k: v.cpu().clone() for k, v in modele.state_dict().items()}
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"[{nom}] arrêt anticipé à l'époque {epoch}.")
            break

    temps_total = time.perf_counter() - debut
    modele.load_state_dict(meilleur_etat)

    # instabilite : ecart-type des variations de perte de validation d'une epoque a l'autre
    variations = np.diff(historique["val_loss"])
    instabilite = float(np.std(variations)) if len(variations) > 1 else 0.0

    return {
        "nom": nom, "modele": modele, "historique": historique,
        "temps_total": temps_total, "accuracy_finale": max(historique["val_acc"]),
        "instabilite_perte_val": instabilite,
    }


## 7. Avant correction : `BatchNorm1d`, lot de 4

C'est le montage de la phase 6 tel quel, seul le lot change.

In [ ]:
resultat_avant = entrainer_et_mesurer("avant_correction_batchnorm_lot4", fabrique_batchnorm, BATCH_SIZE_PANNE)


## 8. Repère : le même sous-échantillon, `BatchNorm1d`, lot de la phase 6 (512)

Pour savoir si l'instabilité observée vient vraiment de la taille de lot et pas seulement du
sous-échantillonnage, on rejoue la configuration de référence (BatchNorm, gros lot) sur ces mêmes
3000 exemples.

In [ ]:
resultat_repere_gros_lot = entrainer_et_mesurer("repere_batchnorm_lot512", fabrique_batchnorm, BATCH_SIZE_PHASE6)


## 9. Après correction : `GroupNorm`, lot de 4

In [ ]:
resultat_apres = entrainer_et_mesurer("apres_correction_groupnorm_lot4", fabrique_groupnorm, BATCH_SIZE_PANNE)


## 10. Le montage corrigé, relancé à la taille de lot de la phase 6

Le Conseil veut savoir si la correction coûte quelque chose quand la machine va bien.

In [ ]:
resultat_apres_gros_lot = entrainer_et_mesurer("apres_correction_groupnorm_lot512", fabrique_groupnorm, BATCH_SIZE_PHASE6)


## 11. Figure : avant / après, au lot de 4

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(resultat_avant["historique"]["epoch"], resultat_avant["historique"]["val_loss"],
         marker="o", label="Avant correction (BatchNorm1d, lot=4)")
plt.plot(resultat_apres["historique"]["epoch"], resultat_apres["historique"]["val_loss"],
         marker="o", label="Après correction (GroupNorm, lot=4)")
plt.title("Phase 7 — Perte de validation au lot de 4, avant / après correction")
plt.xlabel("Époque")
plt.ylabel("Cross-entropy loss (validation)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PHASE7_DIR / "avant_apres_lot4.png", dpi=150)
plt.show()


## 12. Figure : le montage corrigé, lot de 4 contre lot de 512

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(resultat_apres["historique"]["epoch"], resultat_apres["historique"]["val_loss"],
         marker="o", label="GroupNorm, lot=4")
plt.plot(resultat_apres_gros_lot["historique"]["epoch"], resultat_apres_gros_lot["historique"]["val_loss"],
         marker="o", label="GroupNorm, lot=512")
plt.title("Phase 7 — Le montage corrigé coûte-t-il quelque chose à gros lot ?")
plt.xlabel("Époque")
plt.ylabel("Cross-entropy loss (validation)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PHASE7_DIR / "corrige_lot4_vs_lot512.png", dpi=150)
plt.show()


## 13. Tableau récapitulatif

In [ ]:
tableau_phase7 = pd.DataFrame([
    {
        "essai": r["nom"],
        "accuracy_finale": r["accuracy_finale"],
        "instabilite_perte_val": r["instabilite_perte_val"],
        "temps_total_secondes": r["temps_total"],
    }
    for r in [resultat_avant, resultat_repere_gros_lot, resultat_apres, resultat_apres_gros_lot]
])
tableau_phase7


## 14. Ce qui dépendait des autres relevés du lot, et n'aurait jamais dû en dépendre

`BatchNorm1d` normalise chaque canal en utilisant la moyenne et la variance calculées sur **tout le
lot courant** (agrégées ici sur le lot et sur les positions de la séquence). La représentation
produite pour un relevé donné dépendait donc, à l'entraînement, des statistiques des *autres* relevés
tirés dans le même lot — un couplage qui n'a aucun sens : la forme observée dans un témoignage ne
devrait jamais dépendre de quels autres témoignages ont été piochés à ses côtés. `GroupNorm` calcule
ses statistiques **par exemple**, en regroupant les canaux entre eux au lieu de regrouper les exemples
entre eux : la sortie pour un relevé ne dépend plus que de ce relevé.

**Et pour prédire sur un seul relevé ?** À l'évaluation, le modèle est en mode `eval()` (la leçon de
la phase 4) : `BatchNorm1d` utilise alors ses statistiques *glissantes*, accumulées pendant tout
l'entraînement, et non les statistiques du lot en cours — prédire sur un lot d'un seul relevé
fonctionne donc, à condition que `eval()` ait bien été appelé. Le vrai danger se situe à
l'**entraînement** avec un lot de taille 1 : la variance d'un seul exemple sur ses propres positions
devient alors l'unique référence de normalisation pour cet exemple, une estimation bien plus fragile
qu'avec quatre relevés. `GroupNorm` n'a pas ce problème : sa définition ne fait jamais intervenir la
taille du lot.

## 15. Export des résultats

In [ ]:
tableau_phase7.to_csv(PHASE7_DIR / "tableau_recapitulatif.csv", index=False)
pd.DataFrame(resultat_avant["historique"]).to_csv(PHASE7_DIR / "historique_avant_correction.csv", index=False)
pd.DataFrame(resultat_apres["historique"]).to_csv(PHASE7_DIR / "historique_apres_correction.csv", index=False)
pd.DataFrame(resultat_repere_gros_lot["historique"]).to_csv(PHASE7_DIR / "historique_repere_gros_lot.csv", index=False)
pd.DataFrame(resultat_apres_gros_lot["historique"]).to_csv(PHASE7_DIR / "historique_apres_gros_lot.csv", index=False)

resume_phase7 = pd.DataFrame([{
    "taille_sous_echantillon_train": TAILLE_SOUS_ECHANTILLON_TRAIN,
    "taille_sous_echantillon_val": TAILLE_SOUS_ECHANTILLON_VAL,
    "accuracy_avant_lot4": resultat_avant["accuracy_finale"],
    "accuracy_repere_lot512": resultat_repere_gros_lot["accuracy_finale"],
    "accuracy_apres_lot4": resultat_apres["accuracy_finale"],
    "accuracy_apres_lot512": resultat_apres_gros_lot["accuracy_finale"],
    "instabilite_avant_lot4": resultat_avant["instabilite_perte_val"],
    "instabilite_repere_lot512": resultat_repere_gros_lot["instabilite_perte_val"],
    "instabilite_apres_lot4": resultat_apres["instabilite_perte_val"],
    "instabilite_apres_lot512": resultat_apres_gros_lot["instabilite_perte_val"],
}])
resume_phase7.to_csv(PHASE7_DIR / "resume_phase7.csv", index=False)

resume_phase7
